# PolyOCR — Colab debug notebook

Full pipeline: **PP-OCRv5 detection → coordinate-based reading order → TrOCR-large recognition**.

**Before running:** `Runtime → Change runtime type → T4 GPU` (recognition is ~10× faster; the notebook also works on CPU).

Run cells top to bottom. Section 5 lets you upload your own photos; sections 6–7 are the
ordering diagnostics — they show why the *old* app's order changed from photo to photo and
prove the new ordering is independent of detector output order.


## 1. Install dependencies (~2 min)

In [ ]:
%pip install -q "paddleocr>=3.0" paddlepaddle "transformers>=4.45" safetensors
import paddleocr, transformers, torch
print('paddleocr', paddleocr.__version__, '| transformers', transformers.__version__,
      '| torch', torch.__version__, '| cuda:', torch.cuda.is_available())

## 2. Pipeline source (geometry / reading order — pure numpy+cv2)

In [ ]:
%%writefile pipeline.py
"""Pure geometry for the OCR pipeline: reading order, line clustering, cropping.

Everything in this module is deterministic and depends only on numpy/cv2,
so it can be unit-tested without loading any model.

Coordinate convention: image coordinates, x right, y DOWN. Quads are 4x2
arrays ordered [top-left, top-right, bottom-right, bottom-left] after
passing through `order_points`.

Reading-order algorithm (per page):
  1. Estimate global skew as the median angle of the detected boxes'
     horizontal edges (wide boxes only, when enough exist).
  2. Rotate all box corners by the negative skew -> "deskewed space" where
     text lines are horizontal. Only coordinates are rotated, never pixels.
  3. Conservatively split into column blocks: a vertical cut requires a
     projection gap wider than `gap_factor * median box height` with no box
     crossing it, enough boxes on both sides, and both sides spanning most
     of the block height. Recurses (depth-limited) for 3+ columns.
  4. Within each block, cluster boxes into lines: a box joins the line whose
     vertical band it overlaps by >= `overlap_thresh` of the smaller height.
  5. Order lines by band center y; order boxes within a line by center x.
  6. Optionally split very long lines into chunks whose aspect ratio TrOCR
     can handle (it squeezes every crop to a 384x384 square).
"""

from __future__ import annotations

from dataclasses import dataclass, field

import cv2
import numpy as np

__all__ = [
    "order_points",
    "quad_size",
    "estimate_skew",
    "rotate_points",
    "split_columns",
    "cluster_lines",
    "reading_order",
    "chunk_line",
    "merge_quads",
    "expand_quad",
    "perspective_crop",
    "assemble_text",
    "annotate",
    "compose_transcript",
    "Line",
]


# --------------------------------------------------------------------------
# basic quad utilities
# --------------------------------------------------------------------------

def order_points(quad) -> np.ndarray:
    """Return the 4 points ordered [TL, TR, BR, BL].

    Valid for quads rotated less than ~45 degrees, which is all this
    pipeline handles (page-level 90/180 rotation is an upstream
    orientation problem, not a reading-order problem).
    """
    pts = np.asarray(quad, dtype=np.float64).reshape(4, 2)
    idx = np.lexsort((pts[:, 1], pts[:, 0]))  # by x, ties by y
    left, right = pts[idx[:2]], pts[idx[2:]]
    tl, bl = left[np.argsort(left[:, 1])]
    tr, br = right[np.argsort(right[:, 1])]
    return np.array([tl, tr, br, bl])


def quad_size(quad) -> tuple[float, float]:
    """(width, height) as the mean of opposite edge lengths of an ordered quad."""
    q = np.asarray(quad, dtype=np.float64)
    w = (np.linalg.norm(q[1] - q[0]) + np.linalg.norm(q[2] - q[3])) / 2.0
    h = (np.linalg.norm(q[3] - q[0]) + np.linalg.norm(q[2] - q[1])) / 2.0
    return float(w), float(h)


def rotate_points(pts: np.ndarray, theta_deg: float) -> np.ndarray:
    """Rotate points by theta_deg around the origin (standard math rotation;
    with y-down image coords a positive theta maps a +theta-skewed page back
    to horizontal when applied as -theta)."""
    t = np.deg2rad(theta_deg)
    c, s = np.cos(t), np.sin(t)
    rot = np.array([[c, -s], [s, c]])
    return np.asarray(pts, dtype=np.float64) @ rot.T


def estimate_skew(quads: list[np.ndarray], max_abs_deg: float = 30.0,
                  wide_factor: float = 1.5) -> float:
    """Median angle (degrees) of box top/bottom edges.

    Prefers wide boxes (width >= wide_factor * height) because their edge
    direction reliably follows the text baseline; a near-square box around a
    single short word carries almost no angle information. Returns 0.0 when
    the estimate exceeds max_abs_deg — that signals a rotated page, which
    coordinate deskew must not attempt to fix.
    """
    angles, is_wide = [], []
    for quad in quads:
        q = order_points(quad)
        w, h = quad_size(q)
        if w < 2 or h < 2:
            continue
        v = ((q[1] - q[0]) + (q[2] - q[3])) / 2.0  # mean of top and bottom edge
        angles.append(np.degrees(np.arctan2(v[1], v[0])))
        is_wide.append(w >= wide_factor * h)
    if not angles:
        return 0.0
    angles = np.array(angles)
    wide = angles[np.array(is_wide)]
    med = float(np.median(wide if wide.size >= 3 else angles))
    return med if abs(med) <= max_abs_deg else 0.0


# --------------------------------------------------------------------------
# reading order
# --------------------------------------------------------------------------

@dataclass
class Line:
    """One text line: member box indices ordered left-to-right, plus the
    line's vertical band [top, bottom] in deskewed coordinates."""
    members: list[int]
    top: float
    bottom: float
    chunks: list[list[int]] = field(default_factory=list)

    @property
    def height(self) -> float:
        return self.bottom - self.top

    @property
    def center_y(self) -> float:
        return (self.top + self.bottom) / 2.0


def _band(quad_d: np.ndarray) -> tuple[float, float]:
    """Vertical band of a deskewed quad: mean of the 2 upper / 2 lower ys.
    Using means (not min/max) makes the band robust to one stray corner."""
    ys = np.sort(np.asarray(quad_d)[:, 1])
    return float(ys[:2].mean()), float(ys[2:].mean())


def split_columns(quads_d: list[np.ndarray], indices: list[int],
                  gap_factor: float = 2.0, min_side: int = 3,
                  min_y_coverage: float = 0.55, _depth: int = 0) -> list[list[int]]:
    """Conservative column split on deskewed quads. Returns groups of indices
    ordered left-to-right; [indices] unchanged when no confident cut exists.

    A cut requires: an x-projection gap (no box crosses it at any y) wider
    than gap_factor * median box height, at least min_side boxes per side,
    and each side spanning >= min_y_coverage of the block's height — so a
    single line with a big word gap can never trigger a phantom column.
    """
    if _depth >= 2 or len(indices) < 2 * min_side:
        return [indices]

    heights = [_band(quads_d[i])[1] - _band(quads_d[i])[0] for i in indices]
    med_h = float(np.median(heights))
    if med_h <= 0:
        return [indices]

    # merge the boxes' x-intervals; gaps between merged intervals are
    # x-ranges no box touches
    spans = sorted((float(quads_d[i][:, 0].min()), float(quads_d[i][:, 0].max()))
                   for i in indices)
    merged = [list(spans[0])]
    tol = 0.25 * med_h
    for lo, hi in spans[1:]:
        if lo <= merged[-1][1] + tol:
            merged[-1][1] = max(merged[-1][1], hi)
        else:
            merged.append([lo, hi])
    if len(merged) < 2:
        return [indices]

    gaps = [(merged[k + 1][0] - merged[k][1], (merged[k][1] + merged[k + 1][0]) / 2.0)
            for k in range(len(merged) - 1)]
    gaps = [g for g in gaps if g[0] >= gap_factor * med_h]
    if not gaps:
        return [indices]
    cut_x = max(gaps)[1]  # widest qualifying gap

    centers_x = {i: float(quads_d[i][:, 0].mean()) for i in indices}
    left = [i for i in indices if centers_x[i] < cut_x]
    right = [i for i in indices if centers_x[i] >= cut_x]
    if len(left) < min_side or len(right) < min_side:
        return [indices]

    def y_extent(group):
        bands = [_band(quads_d[i]) for i in group]
        return max(b for _, b in bands) - min(t for t, _ in bands)

    total = y_extent(indices)
    if total <= 0 or y_extent(left) < min_y_coverage * total \
            or y_extent(right) < min_y_coverage * total:
        return [indices]

    return (split_columns(quads_d, left, gap_factor, min_side, min_y_coverage, _depth + 1)
            + split_columns(quads_d, right, gap_factor, min_side, min_y_coverage, _depth + 1))


def cluster_lines(quads_d: list[np.ndarray], indices: list[int],
                  overlap_thresh: float = 0.4) -> list[Line]:
    """Group deskewed boxes into text lines by vertical-band overlap.

    A box joins the existing line it overlaps most, provided
    intersection / min(box_height, line_height) >= overlap_thresh.
    Normalizing by the smaller height keeps boxes with deep descenders or
    tall capitals (height outliers) attached to their true line.
    """
    items = []
    for i in indices:
        top, bottom = _band(quads_d[i])
        items.append((i, top, bottom))
    items.sort(key=lambda t: ((t[1] + t[2]) / 2.0, t[0]))

    lines: list[Line] = []
    for i, top, bottom in items:
        h = max(bottom - top, 1e-6)
        best, best_ov = None, overlap_thresh
        for line in lines:
            inter = min(bottom, line.bottom) - max(top, line.top)
            ov = inter / max(min(h, line.height), 1e-6)
            if ov >= best_ov:
                best, best_ov = line, ov
        if best is None:
            lines.append(Line(members=[i], top=top, bottom=bottom))
        else:
            n = len(best.members) + 1
            best.members.append(i)
            best.top += (top - best.top) / n        # running mean of bands
            best.bottom += (bottom - best.bottom) / n
    return lines


def reading_order(quads, column_split: bool = True,
                  overlap_thresh: float = 0.4,
                  gap_factor: float = 2.0) -> tuple[list[Line], float]:
    """Full reading order. Returns (lines, skew_deg); lines are in reading
    order, each line's members ordered left-to-right. Indices refer to the
    input quad list."""
    quads = [np.asarray(q, dtype=np.float64).reshape(4, 2) for q in quads]
    if not quads:
        return [], 0.0

    ordered = [order_points(q) for q in quads]
    theta = estimate_skew(ordered)
    deskewed = [rotate_points(q, -theta) for q in ordered]

    all_idx = list(range(len(quads)))
    groups = split_columns(deskewed, all_idx, gap_factor=gap_factor) \
        if column_split else [all_idx]

    result: list[Line] = []
    for group in groups:  # groups arrive left-to-right; read a column fully first
        lines = cluster_lines(deskewed, group, overlap_thresh=overlap_thresh)
        lines.sort(key=lambda l: l.center_y)
        for line in lines:
            line.members.sort(key=lambda i: (float(deskewed[i][:, 0].mean()), i))
        result.extend(lines)
    return result, theta


def chunk_line(line: Line, quads_d: list[np.ndarray],
               aspect_cap: float = 16.0) -> list[list[int]]:
    """Split a line's members (already x-ordered) into chunks whose merged
    width/height stays under aspect_cap.

    TrOCR resizes every crop to a 384x384 square; beyond ~16:1 the glyphs
    get squeezed into unreadability, so very long lines must be recognized
    in pieces.
    """
    chunks: list[list[int]] = []
    cur: list[int] = []
    cur_lo = cur_hi = 0.0
    h = max(line.height, 1e-6)
    for i in line.members:
        lo, hi = float(quads_d[i][:, 0].min()), float(quads_d[i][:, 0].max())
        if not cur:
            cur, cur_lo, cur_hi = [i], lo, hi
            continue
        new_lo, new_hi = min(cur_lo, lo), max(cur_hi, hi)
        if (new_hi - new_lo) / h > aspect_cap:
            chunks.append(cur)
            cur, cur_lo, cur_hi = [i], lo, hi
        else:
            cur, cur_lo, cur_hi = cur + [i], new_lo, new_hi
    if cur:
        chunks.append(cur)
    line.chunks = chunks
    return chunks


# --------------------------------------------------------------------------
# cropping
# --------------------------------------------------------------------------

def merge_quads(quads: list[np.ndarray]) -> np.ndarray:
    """Smallest rotated rectangle covering all quads, as an ordered quad
    (original image space)."""
    pts = np.vstack([np.asarray(q, dtype=np.float64).reshape(-1, 2) for q in quads])
    rect = cv2.minAreaRect(pts.astype(np.float32))
    return order_points(cv2.boxPoints(rect))


def expand_quad(quad: np.ndarray, pad_frac: float = 0.05) -> np.ndarray:
    """Push the ordered quad's corners outward along its own edge directions
    by pad_frac * height. No clipping to the image: the crop uses
    BORDER_REPLICATE, so slightly out-of-image corners are harmless and the
    quad stays a true parallelogram-ish shape."""
    q = np.asarray(quad, dtype=np.float64).copy()
    _, h = quad_size(q)
    pad = pad_frac * max(h, 1.0)
    ux = (q[1] - q[0]) + (q[2] - q[3])
    uy = (q[3] - q[0]) + (q[2] - q[1])
    nx = ux / max(np.linalg.norm(ux), 1e-6)
    ny = uy / max(np.linalg.norm(uy), 1e-6)
    q[0] += -nx * pad - ny * pad
    q[1] += +nx * pad - ny * pad
    q[2] += +nx * pad + ny * pad
    q[3] += -nx * pad + ny * pad
    return q


def perspective_crop(img: np.ndarray, quad, pad_frac: float = 0.05,
                     allow_rot90: bool = False,
                     interp: int = cv2.INTER_LINEAR) -> np.ndarray | None:
    """Rectify one ordered quad into an axis-aligned crop.

    Returns None for degenerate boxes instead of crashing warpPerspective.
    allow_rot90 rotates strongly vertical crops (h >= 2w) upright — pass it
    only for boxes known not to be single tall characters like 'I'.

    pad_frac controls how far the quad is expanded before warping. TrOCR likes a
    small margin (default 0.05) for handwriting ascenders/descenders. PaddleOCR
    recognition models, however, are trained on TIGHT crops (the detector already
    expands each box via its unclip ratio), so pass pad_frac=0.0 and
    interp=cv2.INTER_CUBIC there to match PP-OCR's own `get_rotate_crop_image` —
    extra padding double-expands the box and pulls neighboring glyphs into the
    crop, which hurts recognition on dense pages.
    """
    q = (expand_quad(order_points(quad), pad_frac) if pad_frac else order_points(quad)
         ).astype(np.float32)
    w = int(round(max(np.linalg.norm(q[1] - q[0]), np.linalg.norm(q[2] - q[3]))))
    h = int(round(max(np.linalg.norm(q[3] - q[0]), np.linalg.norm(q[2] - q[1]))))
    if w < 2 or h < 2:
        return None
    dst = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=np.float32)
    m = cv2.getPerspectiveTransform(q, dst)
    warped = cv2.warpPerspective(img, m, (w, h), flags=interp,
                                 borderMode=cv2.BORDER_REPLICATE)
    if allow_rot90 and h >= 2 * w:
        # np.rot90 returns a negative-stride VIEW; torch.from_numpy (in the
        # TrOCR processor) rejects those, so force a contiguous copy.
        warped = np.ascontiguousarray(np.rot90(warped))  # CCW, matching PaddleOCR
    return warped


# --------------------------------------------------------------------------
# output assembly / debug rendering
# --------------------------------------------------------------------------

def assemble_text(lines: list[Line], line_texts: list[str],
                  para_gap_factor: float = 1.0,
                  column_breaks: set[int] | None = None) -> str:
    """Join per-line texts in reading order. Inserts a blank line when the
    vertical gap to the previous line exceeds para_gap_factor * median line
    height (a paragraph break), or at a column boundary."""
    assert len(lines) == len(line_texts)
    if not lines:
        return ""
    med_h = float(np.median([max(l.height, 1e-6) for l in lines]))
    out: list[str] = []
    for k, (line, text) in enumerate(zip(lines, line_texts)):
        if k > 0:
            gap = line.top - lines[k - 1].bottom
            if (column_breaks and k in column_breaks) or gap > para_gap_factor * med_h:
                out.append("")
        out.append(text)
    return "\n".join(out)


def annotate(img_rgb: np.ndarray, quads: list[np.ndarray],
             lines: list[Line]) -> np.ndarray:
    """Debug overlay: each box outlined and numbered with its reading order,
    colored per line. Makes ordering mistakes visible at a glance."""
    out = img_rgb.copy()
    palette = [(46, 204, 113), (52, 152, 219), (231, 76, 60), (241, 196, 15),
               (155, 89, 182), (26, 188, 156), (230, 126, 34), (149, 165, 166)]
    n = 0
    scale = max(out.shape[0], out.shape[1]) / 1500.0
    fs = max(0.5, 0.9 * scale)
    th = max(1, int(round(2 * scale)))
    for li, line in enumerate(lines):
        color = palette[li % len(palette)]
        for i in line.members:
            n += 1
            q = np.asarray(quads[i], dtype=np.int32).reshape(-1, 2)
            cv2.polylines(out, [q], isClosed=True, color=color, thickness=th)
            pos = (int(q[:, 0].min()), max(int(q[:, 1].min()) - 4, 12))
            cv2.putText(out, str(n), pos, cv2.FONT_HERSHEY_SIMPLEX, fs,
                        color, th, cv2.LINE_AA)
    return out


_PALETTE = [(46, 204, 113), (52, 152, 219), (231, 76, 60), (241, 196, 15),
            (155, 89, 182), (26, 188, 156), (230, 126, 34), (192, 57, 43)]


def _load_font(size: int):
    from PIL import ImageFont
    for name in ("DejaVuSans.ttf",
                 "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                 "arial.ttf", r"C:\Windows\Fonts\arial.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except Exception:
            continue
    try:
        return ImageFont.load_default(size)  # Pillow >= 10
    except TypeError:
        return ImageFont.load_default()


def _wrap(draw, text: str, font, max_w: float) -> list[str]:
    words = text.split()
    if not words:
        return [""]
    out, cur = [], words[0]
    for w in words[1:]:
        if draw.textlength(cur + " " + w, font=font) <= max_w:
            cur += " " + w
        else:
            out.append(cur)
            cur = w
    out.append(cur)
    return out


def compose_transcript(img_rgb: np.ndarray, line_boxes: list[list[np.ndarray]],
                       line_texts: list[str], panel_frac: float = 0.6) -> np.ndarray:
    """One image: the page with per-line numbered, color-coded boxes on the
    left, and the matching numbered transcript on the right.

    line_boxes[k] are the detected quads of line k; line_texts[k] is its text.
    The number and color on each box match its transcript entry, so you can
    read the page and the recognized text side by side and immediately see
    any mismatch. Pure numpy/PIL/cv2 — no models involved."""
    from PIL import Image, ImageDraw

    h, w = img_rgb.shape[:2]
    scale = max(h, w) / 1500.0
    th = max(1, int(round(2 * scale)))
    fs_cv = max(0.5, 1.0 * scale)

    # --- left: draw boxes, label each line at its leftmost box ---
    left = img_rgb.copy()
    for li, boxes in enumerate(line_boxes):
        color = _PALETTE[li % len(_PALETTE)]
        anchor = None
        for box in boxes:
            q = np.asarray(box, dtype=np.int32).reshape(-1, 2)
            cv2.polylines(left, [q], isClosed=True, color=color, thickness=th)
            x0, y0 = int(q[:, 0].min()), int(q[:, 1].min())
            if anchor is None or x0 < anchor[0]:
                anchor = (x0, y0)
        if anchor is not None:
            cv2.putText(left, str(li + 1), (anchor[0], max(anchor[1] - 6, 14)),
                        cv2.FONT_HERSHEY_SIMPLEX, fs_cv, color, th, cv2.LINE_AA)

    # --- right: numbered transcript panel ---
    panel_w = max(360, int(w * panel_frac))
    fsize = max(16, int(round(0.020 * h)))
    line_h = int(fsize * 1.35)
    pad = fsize
    font = _load_font(fsize)
    num_font = _load_font(fsize)

    probe = ImageDraw.Draw(Image.new("RGB", (8, 8)))
    indent = int(max((probe.textlength(f"{k + 1}.", font=num_font)
                      for k in range(max(len(line_texts), 1))), default=0) + 0.6 * fsize)
    text_w = max(panel_w - 2 * pad - indent, 8 * fsize)

    wrapped = []
    for li, txt in enumerate(line_texts):
        segs = _wrap(probe, txt if txt.strip() else "(blank)", font, text_w)
        wrapped.append((li, _PALETTE[li % len(_PALETTE)], segs))

    panel_h = pad + sum(len(segs) * line_h + int(0.5 * line_h)
                        for _, _, segs in wrapped) + pad
    canvas_h = max(h, panel_h)

    panel = Image.new("RGB", (panel_w, canvas_h), (255, 255, 255))
    pd = ImageDraw.Draw(panel)
    y = pad
    for li, color, segs in wrapped:
        pd.text((pad, y), f"{li + 1}.", font=num_font, fill=color)
        for seg in segs:
            pd.text((pad + indent, y), seg, font=font, fill=(20, 20, 25))
            y += line_h
        y += int(0.5 * line_h)

    left_canvas = np.full((canvas_h, w, 3), 255, dtype=np.uint8)
    left_canvas[:h, :w] = left
    sep = np.full((canvas_h, max(2, th), 3), 220, dtype=np.uint8)
    return np.concatenate([left_canvas, sep, np.asarray(panel)], axis=1)


## 3. Engine source (detection + recognition glue)

In [ ]:
%%writefile ocr_engine.py
"""Detection + recognition engine: PaddleOCR text detection -> reading order
(pipeline.py) -> TrOCR recognition.

Heavy dependencies (torch/transformers/paddleocr) are imported lazily inside
the components, so this module can be imported cheaply and the recognizer can
be used without paddle installed (and vice versa).
"""

from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import cv2
import numpy as np

import pipeline

MODEL_HUB_ID = "imperiusrex/Handwritten_model"  # re-save of microsoft/trocr-large-handwritten
LOCAL_MODEL_DIR = Path(__file__).resolve().parent / "local_trocr_model"
DET_MODEL_NAME = os.environ.get("OCR_DET_MODEL", "PP-OCRv5_server_det")

# crops more elongated than this get recognized in pieces (TrOCR squeezes
# everything to a 384x384 square)
ASPECT_CAP = 16.0
# a merged line crop whose height exceeds this multiple of the median member
# height probably swallowed two stacked lines -> fall back to per-box crops
MERGE_HEIGHT_GUARD = 1.8
MIN_DET_SCORE = 0.30


def resolve_rec_source() -> str:
    """Prefer the local model dir when it is actually loadable."""
    if (LOCAL_MODEL_DIR / "model.safetensors").is_file() \
            and (LOCAL_MODEL_DIR / "config.json").is_file():
        return str(LOCAL_MODEL_DIR)
    return MODEL_HUB_ID


class Detector:
    """PP-OCRv5 text detection wrapper that returns clean (N,4,2) quads in
    original-image coordinates."""

    def __init__(self, model_name: str = DET_MODEL_NAME,
                 enable_mkldnn: bool | None = None):
        from paddleocr import TextDetection  # lazy: paddle is optional
        self._cls = TextDetection
        self._model_name = model_name
        if enable_mkldnn is None:
            env = os.environ.get("OCR_DET_MKLDNN")
            # paddle's PIR + oneDNN executor is broken on Windows CPU
            # (ConvertPirAttribute2RuntimeAttribute NotImplementedError)
            enable_mkldnn = env == "1" if env is not None else sys.platform != "win32"
        self._det = TextDetection(model_name=model_name, enable_mkldnn=enable_mkldnn)

    def __call__(self, img_rgb: np.ndarray, min_score: float = MIN_DET_SCORE) -> np.ndarray:
        # PaddleOCR consumes cv2-style BGR; gradio/PIL hand us RGB.
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        try:
            results = self._det.predict(img_bgr, batch_size=1)
        except NotImplementedError as e:  # oneDNN executor bug -> retry without it
            if "Pir" not in str(e) and "onednn" not in str(e).lower():
                raise
            self._det = self._cls(model_name=self._model_name, enable_mkldnn=False)
            results = self._det.predict(img_bgr, batch_size=1)

        quads: list[np.ndarray] = []
        for res in results:
            polys = res.get("dt_polys", None) if hasattr(res, "get") else None
            if polys is None:
                continue
            scores = res.get("dt_scores", None)
            for k, poly in enumerate(polys):
                score = float(scores[k]) if scores is not None and k < len(scores) else 1.0
                if score < min_score:
                    continue
                arr = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
                if arr.shape[0] > 4:  # polygon mode -> reduce to min-area quad
                    arr = pipeline.merge_quads([arr])
                if arr.shape[0] != 4:
                    continue
                w, h = pipeline.quad_size(pipeline.order_points(arr))
                if w < 2 or h < 2:
                    continue
                quads.append(arr)
        return np.array(quads, dtype=np.float64) if quads else np.zeros((0, 4, 2))


class Recognizer:
    """Batched TrOCR recognition."""

    def __init__(self, source: str | None = None, device: str | None = None,
                 fp16: bool | None = None):
        import torch
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel

        self.torch = torch
        self.source = source or resolve_rec_source()
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        use_fp16 = fp16 if fp16 is not None else self.device.type == "cuda"
        dtype = torch.float16 if use_fp16 else torch.float32

        self.processor = TrOCRProcessor.from_pretrained(self.source)
        self.model = VisionEncoderDecoderModel.from_pretrained(self.source, torch_dtype=dtype)
        self.model.to(self.device)
        self.model.eval()

    def __call__(self, crops: list[np.ndarray], batch_size: int = 8,
                 num_beams: int = 1, max_new_tokens: int = 96) -> list[str]:
        texts: list[str] = []
        for start in range(0, len(crops), batch_size):
            # ascontiguousarray guards against negative-stride arrays (e.g. from
            # np.rot90/flips upstream), which torch.from_numpy refuses to wrap
            batch = [np.ascontiguousarray(c) for c in crops[start:start + batch_size]]
            pixel_values = self.processor(images=batch, return_tensors="pt").pixel_values
            pixel_values = pixel_values.to(self.device, dtype=self.model.dtype)
            with self.torch.inference_mode():
                ids = self.model.generate(
                    pixel_values,
                    max_new_tokens=max_new_tokens,
                    num_beams=num_beams,
                    early_stopping=num_beams > 1,
                    # the shipped config says use_cache=false (a training
                    # leftover in microsoft's checkpoint); without the KV
                    # cache generation re-runs the whole decoder per token
                    use_cache=True,
                )
            texts.extend(self.processor.batch_decode(ids, skip_special_tokens=True))
        return [t.strip() for t in texts]


class OcrEngine:
    def __init__(self, rec_source: str | None = None, det_model: str = DET_MODEL_NAME,
                 device: str | None = None, fp16: bool | None = None):
        self.detector = Detector(det_model)
        self.recognizer = Recognizer(rec_source, device=device, fp16=fp16)

    def run(self, img_rgb: np.ndarray, merge_segments: bool = True,
            num_beams: int = 1, batch_size: int = 8) -> dict:
        """Full page OCR. Returns dict with text, lines, overlay, skew_deg, timing."""
        t0 = time.perf_counter()
        if img_rgb.ndim == 2:
            img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_GRAY2RGB)
        elif img_rgb.shape[2] == 4:
            img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_RGBA2RGB)

        quads = self.detector(img_rgb)
        t_det = time.perf_counter()

        if len(quads) == 0:
            # fallback: treat the whole image as one line and say so
            text = self.recognizer([img_rgb], num_beams=num_beams)[0]
            return {
                "text": text,
                "lines": [],
                "overlay": img_rgb,
                "composite": pipeline.compose_transcript(img_rgb, [[]], [text]),
                "skew_deg": 0.0,
                "note": "no text boxes detected; whole-image fallback",
                "seconds": {"detect": t_det - t0, "recognize": time.perf_counter() - t_det},
            }

        lines, theta = pipeline.reading_order(quads)
        ordered = [pipeline.order_points(q) for q in quads]
        deskewed = [pipeline.rotate_points(q, -theta) for q in ordered]
        heights = [pipeline.quad_size(q)[1] for q in ordered]
        med_h = float(np.median(heights))

        # build crops: one per line-chunk when merging, else one per box
        crops: list[np.ndarray] = []
        owners: list[int] = []  # line index of each crop
        for li, line in enumerate(lines):
            chunks = pipeline.chunk_line(line, deskewed, aspect_cap=ASPECT_CAP) \
                if merge_segments else [[i] for i in line.members]
            for chunk in chunks:
                crop = None
                if merge_segments:
                    merged = pipeline.merge_quads([ordered[i] for i in chunk])
                    if pipeline.quad_size(merged)[1] <= MERGE_HEIGHT_GUARD * med_h:
                        crop = pipeline.perspective_crop(img_rgb, merged)
                if crop is not None:
                    crops.append(crop)
                    owners.append(li)
                else:  # per-box fallback (merge declined or degenerate)
                    for i in chunk:
                        w, h = pipeline.quad_size(ordered[i])
                        c = pipeline.perspective_crop(
                            img_rgb, ordered[i],
                            allow_rot90=h > 2.2 * med_h)
                        if c is not None:
                            crops.append(c)
                            owners.append(li)

        chunk_texts = self.recognizer(crops, batch_size=batch_size, num_beams=num_beams)
        line_texts = ["" for _ in lines]
        for text, li in zip(chunk_texts, owners):
            line_texts[li] = (line_texts[li] + " " + text).strip()

        text = pipeline.assemble_text(lines, line_texts)
        overlay = pipeline.annotate(img_rgb, quads, lines)
        line_boxes = [[ordered[i] for i in lines[k].members] for k in range(len(lines))]
        composite = pipeline.compose_transcript(img_rgb, line_boxes, line_texts)
        t_rec = time.perf_counter()

        return {
            "text": text,
            "lines": [
                {"text": line_texts[k], "boxes": [quads[i].tolist() for i in lines[k].members]}
                for k in range(len(lines))
            ],
            "overlay": overlay,
            "composite": composite,
            "skew_deg": theta,
            "seconds": {"detect": t_det - t0, "recognize": t_rec - t_det},
        }


## 4. Geometry unit tests (no models needed, ~1 s)
20 tests incl. a fixture where naive y-sorting provably fails.

In [ ]:
%%writefile test_geometry.py
"""Unit tests for pipeline.py (pure geometry — no models needed).

Run:  python -m unittest discover -s tests -v
"""

import sys
import unittest
from pathlib import Path

import cv2
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import pipeline  # noqa: E402


def make_box(x, y, w, h):
    """Axis-aligned quad [TL, TR, BR, BL]."""
    return np.array([[x, y], [x + w, y], [x + w, y + h], [x, y + h]], dtype=np.float64)


def rotate_page(quads, theta_deg, center=(0, 0)):
    """Rotate quads by +theta around center — simulates a skewed photo."""
    t = np.deg2rad(theta_deg)
    c, s = np.cos(t), np.sin(t)
    rot = np.array([[c, -s], [s, c]])
    ctr = np.asarray(center, dtype=np.float64)
    return [(np.asarray(q) - ctr) @ rot.T + ctr for q in quads]


def flat_order(lines):
    return [i for line in lines for i in line.members]


class TestOrderPoints(unittest.TestCase):
    def test_scrambled_square(self):
        quad = [[100, 10], [10, 10], [10, 50], [100, 50]]  # TR, TL, BL, BR
        out = pipeline.order_points(quad)
        np.testing.assert_allclose(
            out, [[10, 10], [100, 10], [100, 50], [10, 50]])

    def test_slightly_rotated(self):
        quad = make_box(50, 50, 200, 40)
        (rotated,) = rotate_page([quad], 10, center=(150, 70))
        out = pipeline.order_points(rotated[[2, 0, 3, 1]])  # scramble rows
        np.testing.assert_allclose(out, rotated, atol=1e-9)


class TestReadingOrder(unittest.TestCase):
    def test_shuffled_grid(self):
        # 3 lines x 3 words, fed in shuffled order
        quads, expected = [], []
        for r in range(3):
            for c in range(3):
                quads.append(make_box(50 + 220 * c, 40 + 70 * r, 180, 40))
        rng = np.random.default_rng(7)
        perm = rng.permutation(9)
        shuffled = [quads[i] for i in perm]
        lines, theta = pipeline.reading_order(shuffled)
        self.assertEqual(len(lines), 3)
        self.assertAlmostEqual(theta, 0.0, places=5)
        # mapping back: reading order must equal row-major original order
        recovered = [int(perm[i]) for i in flat_order(lines)]
        self.assertEqual(recovered, list(range(9)))

    def test_skew_where_naive_y_sort_fails(self):
        # Long lines + 8 deg skew: the rightmost word of line 1 sits LOWER in
        # the photo than the leftmost word of line 2, so sorting boxes by
        # center-y interleaves the lines. Deskewed clustering must not.
        quads = []
        for r in range(3):           # 3 lines
            for c in range(5):       # 5 words, total width ~1100 px
                quads.append(make_box(40 + 220 * c, 100 + 70 * r, 180, 40))
        skewed = rotate_page(quads, 8, center=(600, 200))

        # sanity: prove the naive method actually fails on this input
        centers = [np.mean(q, axis=0) for q in skewed]
        naive = sorted(range(15), key=lambda i: (centers[i][1], centers[i][0]))
        self.assertNotEqual(naive, list(range(15)),
                            "test fixture too easy: naive sort accidentally works")

        lines, theta = pipeline.reading_order(skewed)
        self.assertEqual(len(lines), 3)
        self.assertAlmostEqual(theta, 8.0, delta=0.5)
        self.assertEqual(flat_order(lines), list(range(15)))

    def test_two_columns(self):
        # 2 columns x 4 lines; column gap (160 px) >> box height (30 px)
        quads, left_idx, right_idx = [], [], []
        for col, x0 in enumerate((40, 500)):
            for r in range(4):
                (left_idx if col == 0 else right_idx).append(len(quads))
                quads.append(make_box(x0, 50 + 60 * r, 300, 30))
        lines, _ = pipeline.reading_order(quads)
        order = flat_order(lines)
        self.assertEqual(order[:4], left_idx, "left column must be read first")
        self.assertEqual(order[4:], right_idx)

    def test_single_column_with_wide_word_gap_not_split(self):
        # One line has a huge inner gap; other lines bridge that x-range, so
        # no column split may happen.
        quads = [
            make_box(40, 50, 120, 30), make_box(600, 50, 120, 30),  # gappy line
            make_box(40, 110, 680, 30),                              # full-width
            make_box(40, 170, 680, 30),
            make_box(40, 230, 120, 30), make_box(600, 230, 120, 30),
        ]
        lines, _ = pipeline.reading_order(quads)
        self.assertEqual(len(lines), 4)
        self.assertEqual(flat_order(lines), [0, 1, 2, 3, 4, 5])

    def test_descender_height_outlier_same_line(self):
        # box B is 45% taller (deep descenders) but clearly the same line;
        # box C starts where B's descenders reach but overlaps B only 10px
        a = make_box(40, 100, 200, 40)    # band 100..140
        b = make_box(260, 102, 200, 58)   # band 102..160 (descenders)
        c = make_box(40, 150, 200, 40)    # band 150..190 -> next line
        lines, _ = pipeline.reading_order([a, b, c])
        self.assertEqual(len(lines), 2)
        self.assertEqual(lines[0].members, [0, 1])
        self.assertEqual(lines[1].members, [2])

    def test_empty(self):
        lines, theta = pipeline.reading_order([])
        self.assertEqual(lines, [])
        self.assertEqual(theta, 0.0)

    def test_single_box(self):
        lines, _ = pipeline.reading_order([make_box(10, 10, 100, 30)])
        self.assertEqual(flat_order(lines), [0])


class TestChunking(unittest.TestCase):
    def test_long_line_is_chunked(self):
        # 10 adjacent words, 100x30 each -> merged aspect 1000/30 = 33 > 16
        quads = [make_box(40 + 100 * c, 50, 96, 30) for c in range(10)]
        lines, _ = pipeline.reading_order(quads)
        self.assertEqual(len(lines), 1)
        deskewed = [pipeline.order_points(q) for q in quads]
        chunks = pipeline.chunk_line(lines[0], deskewed, aspect_cap=16.0)
        self.assertGreaterEqual(len(chunks), 2)
        self.assertEqual([i for ch in chunks for i in ch], list(range(10)),
                         "chunking must preserve order")
        for ch in chunks:
            lo = min(deskewed[i][:, 0].min() for i in ch)
            hi = max(deskewed[i][:, 0].max() for i in ch)
            self.assertLessEqual((hi - lo) / lines[0].height, 16.0 + 1e-6)

    def test_short_line_single_chunk(self):
        quads = [make_box(40, 50, 200, 40), make_box(260, 50, 200, 40)]
        lines, _ = pipeline.reading_order(quads)
        deskewed = [pipeline.order_points(q) for q in quads]
        chunks = pipeline.chunk_line(lines[0], deskewed)
        self.assertEqual(chunks, [[0, 1]])


class TestMergeAndCrop(unittest.TestCase):
    def test_merge_quads_covers_members_tightly(self):
        words = [make_box(40 + 150 * c, 100, 130, 36) for c in range(4)]
        skewed = rotate_page(words, 5, center=(300, 118))
        merged = pipeline.merge_quads(skewed)
        w, h = pipeline.quad_size(merged)
        self.assertGreater(w, 550)            # spans all words
        self.assertLess(h, 36 * 1.4)          # but stays one line tall
        # every member corner inside (or on) the merged rect
        merged_i = merged.astype(np.float32)
        for q in skewed:
            for pt in q:
                d = cv2.pointPolygonTest(merged_i, (float(pt[0]), float(pt[1])), True)
                self.assertGreater(d, -1.5)

    def test_crop_rectifies_rotated_text(self):
        # paint a black bar on white, rotated 12 deg; crop must come back
        # axis-aligned with dark pixels in the middle row, white at corners
        img = np.full((400, 600, 3), 255, dtype=np.uint8)
        quad = make_box(150, 180, 300, 40)
        (rq,) = rotate_page([quad], 12, center=(300, 200))
        cv2.fillPoly(img, [rq.astype(np.int32)], (0, 0, 0))
        crop = pipeline.perspective_crop(img, rq, pad_frac=0.10)
        self.assertIsNotNone(crop)
        ch, cw = crop.shape[:2]
        self.assertGreater(cw / ch, 4.0)  # wide line stays wide
        mid = crop[ch // 2, cw // 2].mean()
        corner = crop[1, 1].mean()
        self.assertLess(mid, 60)          # painted bar
        self.assertGreater(corner, 200)   # padded margin is page-white

    def test_tight_crop_matches_box_and_excludes_neighbor(self):
        # a tight crop (pad_frac=0, PP-OCR style) must size to the box itself and
        # NOT include a neighbor glyph just outside it; a padded crop drags it in.
        img = np.full((200, 400, 3), 255, dtype=np.uint8)
        box = make_box(100, 80, 120, 40)            # the text box
        cv2.fillPoly(img, [make_box(226, 80, 20, 40).astype(np.int32)], (0, 0, 0))  # neighbor to the right
        tight = pipeline.perspective_crop(img, box, pad_frac=0.0)
        padded = pipeline.perspective_crop(img, box, pad_frac=0.25)
        self.assertEqual(tight.shape[1], 120)        # exact box width, no expansion
        self.assertGreater(padded.shape[1], tight.shape[1])  # padding enlarges
        # tight crop's right edge is page-white (neighbor excluded); padded pulls it in
        self.assertGreater(tight[:, -2].mean(), 200)
        self.assertLess(padded[:, -2].mean(), tight[:, -2].mean())

    def test_degenerate_quad_returns_none(self):
        line = np.array([[10, 10], [200, 10], [200, 10], [10, 10]], dtype=np.float64)
        self.assertIsNone(pipeline.perspective_crop(
            np.zeros((100, 300, 3), np.uint8), line))

    def test_vertical_crop_rot90(self):
        img = np.zeros((300, 200, 3), np.uint8)
        tall = make_box(50, 30, 40, 200)
        upright = pipeline.perspective_crop(img, tall, allow_rot90=True)
        self.assertGreater(upright.shape[1], upright.shape[0])
        self.assertTrue(upright.flags["C_CONTIGUOUS"],
                        "rot90 crop must be contiguous for torch.from_numpy")
        kept = pipeline.perspective_crop(img, tall, allow_rot90=False)
        self.assertGreater(kept.shape[0], kept.shape[1])


class TestAssemble(unittest.TestCase):
    def test_paragraph_gap(self):
        l1 = pipeline.Line([0], top=100, bottom=140)
        l2 = pipeline.Line([1], top=150, bottom=190)   # gap 10 -> same para
        l3 = pipeline.Line([2], top=260, bottom=300)   # gap 70 > med_h 40
        text = pipeline.assemble_text([l1, l2, l3], ["one", "two", "three"])
        self.assertEqual(text, "one\ntwo\n\nthree")

    def test_empty(self):
        self.assertEqual(pipeline.assemble_text([], []), "")


class TestComposeTranscript(unittest.TestCase):
    def test_composite_shape_and_no_mutation(self):
        img = np.full((300, 500, 3), 255, np.uint8)
        before = img.copy()
        boxes = [[make_box(20, 30, 200, 30)], [make_box(20, 90, 260, 30)]]
        out = pipeline.compose_transcript(img, boxes, ["first line", "second line"])
        self.assertTrue(np.array_equal(img, before), "must not mutate input")
        self.assertGreaterEqual(out.shape[1], img.shape[1], "panel widens the image")
        self.assertGreaterEqual(out.shape[0], img.shape[0])
        self.assertEqual(out.shape[2], 3)
        # the right-hand panel must contain dark (text) pixels on white
        panel = out[:, img.shape[1]:]
        self.assertTrue((panel < 80).any(), "transcript text not rendered")

    def test_handles_blank_text_and_empty_boxes(self):
        img = np.full((120, 200, 3), 255, np.uint8)
        out = pipeline.compose_transcript(img, [[]], [""])
        self.assertEqual(out.shape[2], 3)
        self.assertGreaterEqual(out.shape[1], img.shape[1])


class TestAnnotate(unittest.TestCase):
    def test_overlay_draws_in_order(self):
        img = np.full((200, 400, 3), 255, np.uint8)
        quads = [make_box(20, 30, 100, 30), make_box(150, 30, 100, 30)]
        lines, _ = pipeline.reading_order(quads)
        out = pipeline.annotate(img, quads, lines)
        self.assertEqual(out.shape, img.shape)
        self.assertFalse(np.array_equal(out, img), "overlay must draw something")
        self.assertTrue(np.array_equal(img, np.full((200, 400, 3), 255, np.uint8)),
                        "annotate must not mutate the input image")


if __name__ == "__main__":
    unittest.main(verbosity=2)


In [ ]:
!python -m unittest test_geometry -v

## 5. Load models and run on your own photos
Loads TrOCR from the Hub (`imperiusrex/Handwritten_model`, ~2.2 GB, fp16 on GPU) and
PP-OCRv5 server detection.

**Giving it photos — set `IMAGE_SOURCE` in the cell:**
- `"ask"` (default): every run opens a *Choose Files* dialog and uses **only the files you
  pick that run**, so testing a new image never silently reuses the previous one.
- `"folder"`: open the 📁 sidebar, drag images into the **`inputs`** folder, then re-run;
  it reads whatever is in that folder (handy for batches).

**Output:** the recognized text is **printed right here inline** (no download needed), and
the result image — the page with numbered boxes on the left and the matching numbered
transcript beside it — is shown below each photo. Section 8 can zip everything if you want
the files too.

In [ ]:
import os
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'
from ocr_engine import OcrEngine

MERGE_SEGMENTS = True   # merge same-line fragments into one crop (recommended)
NUM_BEAMS = 1           # try 4 for slightly better accuracy, slower

engine = OcrEngine()
print('TrOCR on', engine.recognizer.device, '| dtype', engine.recognizer.model.dtype)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Where images come from each run:
#   'ask'    -> always open the upload dialog and use ONLY the files you pick now
#               (so a previously-tested image is never silently reused)
#   'folder' -> read images from INPUT_DIR; drag files into that sidebar folder
IMAGE_SOURCE = 'ask'
INPUT_DIR = '/content/inputs'
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def load_images():
    if IMAGE_SOURCE == 'folder':
        d = Path(INPUT_DIR); d.mkdir(parents=True, exist_ok=True)
        paths = sorted(p for p in d.rglob('*')
                       if p.suffix.lower() in EXTS
                       and '_panel' not in p.stem and '_overlay' not in p.stem)
        if not paths:
            print(f"No images in {INPUT_DIR}. Open the sidebar (folder icon), "
                  f"drag images into 'inputs', and re-run.")
        else:
            print(f'Using {len(paths)} image(s) from {INPUT_DIR}:')
            for p in paths: print('  ', p.name)
        return {p.name: p.read_bytes() for p in paths}
    from google.colab import files          # 'ask' (default)
    print('Pick image(s) — only the files you select now are used:')
    return files.upload()

images = load_images()

results = {}   # reset every run, so an earlier image never lingers
for name, data in images.items():
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        print(f'!! could not decode {name}'); continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = engine.run(img, merge_segments=MERGE_SEGMENTS, num_beams=NUM_BEAMS)
    results[name] = (img, res)
    # main visual: page with numbered boxes + matching transcript beside it
    comp = res['composite']
    ch, cw = comp.shape[:2]
    plt.figure(figsize=(18, max(5, 18 * ch / cw)))
    plt.imshow(comp); plt.axis('off')
    plt.title(f"{name} — page + transcript (numbers match)"); plt.show()
    print(f"--- {name} | skew {res['skew_deg']:.1f}° | det {res['seconds']['detect']:.1f}s"
          f" | rec {res['seconds']['recognize']:.1f}s ---")
    if res.get('note'): print('[', res['note'], ']')
    print(res['text'])   # also printed inline as plain copy-pasteable text
    print()

## 6. Why the old app's order differed per photo
The old `app.py` used the **detector's output order, reversed** (`cropped_images.reverse()`).
That order is an undefined implementation detail of `cv2.findContours`, so it changes with
image content — right on some photos, scrambled on others. Left: old behavior. Right: new
coordinate-based order. (Uses the last photo you uploaded.)

In [ ]:
import pipeline

name, (img, _res) = list(results.items())[-1]
quads = engine.detector(img)
print(f'{name}: {len(quads)} boxes detected')

legacy_lines = [pipeline.Line(members=[i], top=0.0, bottom=0.0)
                for i in reversed(range(len(quads)))]   # what legacy_app.py did
new_lines, theta = pipeline.reading_order(quads)

fig, axes = plt.subplots(1, 2, figsize=(22, 11))
axes[0].imshow(pipeline.annotate(img, list(quads), legacy_lines))
axes[0].set_title('OLD: detector order, reversed'); axes[0].axis('off')
axes[1].imshow(pipeline.annotate(img, list(quads), new_lines))
axes[1].set_title(f'NEW: coordinate-based (skew {theta:.1f}°)'); axes[1].axis('off')
plt.show()

## 7. Proof: new ordering is independent of detector output order
We shuffle the detected boxes randomly 5 times and recompute the reading order. The
resulting sequence of boxes (identified by centroid) must be identical every time —
i.e. the failure mode you saw (order changing run to run / photo to photo) cannot occur.

In [ ]:
import random

def order_signature(quad_list):
    lines, _ = pipeline.reading_order(quad_list)
    return [tuple(np.round(np.asarray(quad_list[i]).mean(axis=0), 1))
            for line in lines for i in line.members]

base = order_signature(list(quads))
for seed in range(5):
    shuffled = list(quads)
    random.Random(seed).shuffle(shuffled)
    assert order_signature(shuffled) == base, f'order changed under shuffle (seed {seed})!'
print('PASS: reading order identical under 5 random shuffles of detector output')

## 8. Download results (texts + overlays as a zip)

In [ ]:
import zipfile
from pathlib import Path

out = Path('ocr_results.zip')
with zipfile.ZipFile(out, 'w') as z:
    for name, (img, res) in results.items():
        stem = Path(name).stem
        z.writestr(f'{stem}_ocr.txt', res['text'])
        for tag, key in (('panel', 'composite'), ('overlay', 'overlay')):
            ok, buf = cv2.imencode('.png', cv2.cvtColor(res[key], cv2.COLOR_RGB2BGR))
            if ok: z.writestr(f'{stem}_{tag}.png', buf.tobytes())
from google.colab import files
files.download(str(out))

## 9. Measure accuracy (CER / WER) and compare against other OCRs
This is how you answer *"does my pipeline beat anything?"* — every system is scored on the
**same** images, so the numbers are comparable.

**Metrics:**
- **CER / WER** (lower is better) — character / word error rate vs. ground truth
  (corpus-aggregated, the standard way; can exceed 100% if a system hallucinates). The headline.
- **WordAcc** (higher is better) — order-free word accuracy: fraction of ground-truth words
  present in the prediction regardless of position. Separates recognition from reading order
  cleanly (high WordAcc + high CER ⇒ ordering/segmentation issue; low WordAcc ⇒ recognition issue).

**You need ground truth.** Put each image and its correct transcript together: for `foo.jpg`,
a sibling `foo.txt` with one physical line per line, in reading order. Drag both into the
`eval_data` folder (sidebar). Start with **5–10 of your own pages** — even that gives signal.
For a standard benchmark, see the note at the bottom (IAM / GNHK).

In [ ]:
%%writefile evaluate.py
"""OCR evaluation harness: CER / WER + an order-free recognition diagnostic,
plus baseline runners so several systems are scored on the *same* data.

Metrics
-------
- **CER** (Character Error Rate) = Levenshtein(ref, hyp) / len(ref), aggregated
  at corpus level (sum of edits / sum of ref chars — the standard way). Lower
  is better; can exceed 100% when a system hallucinates. This is the headline.
- **WER** (Word Error Rate) = same, on whitespace tokens.
- **WordAcc** = order-free word accuracy = fraction of reference words present
  in the prediction regardless of position (multiset intersection / n_ref).
  Higher is better. This separates recognition from reading order WITHOUT the
  paradoxes of a char-level "order-invariant CER": sorting words/chars to remove
  order misaligns recognition errors and can exceed the document CER, so such a
  decomposition is ill-posed on real data. WordAcc is a clean, bounded [0,1]
  signal — high WordAcc with high CER => ordering/segmentation issue; low
  WordAcc => genuine recognition issue. It is NOT subtracted from CER.

A "system" is just a function image_rgb -> text (and optionally -> list of
lines, for the order diagnostic). Their pipeline, Tesseract, EasyOCR and
PaddleOCR are all wired up below behind lazy imports.

CLI:
    # your own labeled folder
    python evaluate.py --data DIR
    python evaluate.py --data DIR --baselines tesseract,easyocr,paddleocr --csv out.csv
    # a public dataset (Hugging Face) — IAM test lines, quick 200-sample check:
    python evaluate.py --hf iam-lines --n 200
    python evaluate.py --hf iam-lines            # full IAM test (2,920 lines)

--data layout: each image `foo.jpg` has its ground-truth transcript in a sibling
text file `foo.txt` (or `foo.gt.txt`), one physical line per line, in reading
order. Feed page images for document-level scores, or single-line crops for
line-level scores.

--hf: a preset in HF_DATASETS (currently 'iam-lines' = Teklia/IAM-line) or any
hub dataset id with an image column + a text column (auto-detected). Line-level
datasets are scored on the recognizer alone (no detection/ordering), directly
comparable to published TrOCR CER (~2.9% on IAM test).
"""

from __future__ import annotations

import argparse
import re
import string
import time
from dataclasses import dataclass, field
from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


# --------------------------------------------------------------------------
# edit distance + normalization
# --------------------------------------------------------------------------

def levenshtein(a, b) -> int:
    """Edit distance between two sequences (strings for CER, token lists for WER)."""
    if a == b:
        return 0
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        ai = a[i - 1]
        for j in range(1, lb + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[lb]


def normalize(s: str, lowercase: bool = False, strip_punct: bool = False,
              collapse_ws: bool = True) -> str:
    """Text normalization applied to BOTH ref and hyp before scoring.

    Default collapses runs of whitespace (incl. newlines) to single spaces so
    detection/segmentation formatting differences don't inflate CER, while
    leaving case and punctuation intact (standard CER). Flags let you relax it.
    """
    if lowercase:
        s = s.lower()
    if strip_punct:
        s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s).strip() if collapse_ws else s.strip()
    return s


@dataclass
class NormCfg:
    lowercase: bool = False
    strip_punct: bool = False
    collapse_ws: bool = True

    def __call__(self, s: str) -> str:
        return normalize(s, self.lowercase, self.strip_punct, self.collapse_ws)


def cer_counts(ref: str, hyp: str, norm: NormCfg) -> tuple[int, int]:
    r, h = norm(ref), norm(hyp)
    return levenshtein(r, h), len(r)


def wer_counts(ref: str, hyp: str, norm: NormCfg) -> tuple[int, int]:
    r, h = norm(ref).split(), norm(hyp).split()
    return levenshtein(r, h), len(r)


def word_acc_counts(ref: str, hyp: str, norm: NormCfg) -> tuple[int, int]:
    """Order-free recognition signal: (matched, n_ref_words) where `matched` is
    the multiset intersection of reference and predicted words.

    word accuracy = matched / n_ref_words = fraction of ground-truth words the
    system produced *regardless of position*. This isolates recognition quality
    from reading order without the paradoxes of a char-level "order-invariant
    CER" (sorting characters/words misaligns recognition errors and can exceed
    the document CER). High word-accuracy with high CER => ordering/segmentation
    problem; low word-accuracy => genuine recognition problem."""
    from collections import Counter
    r = Counter(norm(ref).split())
    h = Counter(norm(hyp).split())
    matched = sum((r & h).values())
    return matched, sum(r.values())


# --------------------------------------------------------------------------
# dataset
# --------------------------------------------------------------------------

@dataclass
class Sample:
    name: str
    gt: str
    image_path: Path | None = None
    image: object | None = None  # in-memory PIL.Image or RGB np.ndarray (optional)

    def image_rgb(self):
        import cv2
        import numpy as np
        if self.image is not None:
            arr = np.asarray(self.image)
            if arr.ndim == 2:
                arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
            elif arr.shape[2] == 4:
                arr = cv2.cvtColor(arr, cv2.COLOR_RGBA2RGB)
            return np.ascontiguousarray(arr)
        bgr = cv2.imdecode(np.fromfile(self.image_path, np.uint8), cv2.IMREAD_COLOR)
        if bgr is None:
            raise ValueError(f"could not decode {self.image_path}")
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def load_pairs(folder: str | Path) -> list[Sample]:
    """Local folder: each image has a sibling .txt / .gt.txt ground truth."""
    folder = Path(folder)
    samples: list[Sample] = []
    for img in sorted(folder.rglob("*")):
        if img.suffix.lower() not in IMAGE_EXTS:
            continue
        if any(t in img.stem for t in ("_panel", "_overlay")):  # skip our outputs
            continue
        gt_path = None
        for cand in (img.with_suffix(".gt.txt"), img.with_suffix(".txt"),
                     img.with_name(img.stem + ".gt.txt")):
            if cand.is_file():
                gt_path = cand
                break
        if gt_path is None:
            print(f"  (skip {img.name}: no ground-truth .txt beside it)")
            continue
        samples.append(Sample(name=img.name, image_path=img,
                              gt=gt_path.read_text(encoding="utf-8")))
    return samples


# verified-good presets; `level='line'` -> score the recognizer directly,
# `level='page'` -> score the full detection+ordering+recognition pipeline.
HF_DATASETS = {
    "iam-lines": dict(path="Teklia/IAM-line", image_col="image", text_col="text",
                      split="test", level="line",
                      note="IAM test lines; compare to TrOCR-large CER ~2.9%"),
    "iam-sentences": dict(path="alpayariyak/IAM_Sentences", image_col="image",
                          text_col="text", split="train", level="page",
                          note="IAM sentences as MULTI-LINE images; tests the full "
                               "pipeline. Recognition CER is optimistic (TrOCR trained "
                               "on IAM); detection+ordering test is valid."),
}

_IMG_COL_HINTS = ("image", "img", "im", "picture")
_TXT_COL_HINTS = ("text", "label", "transcription", "transcript", "sentence",
                  "gt", "ground_truth", "caption")


def _auto_col(row: dict, hints) -> str:
    keys = list(row.keys())
    for h in hints:
        for k in keys:
            if k.lower() == h:
                return k
    for h in hints:
        for k in keys:
            if h in k.lower():
                return k
    raise KeyError(f"could not auto-detect column from {keys} (hints={hints}); "
                   f"pass image_col=/text_col= explicitly")


def load_hf(name_or_path: str, split: str | None = None, n: int | None = None,
            image_col: str | None = None, text_col: str | None = None,
            config: str | None = None, streaming: bool = True) -> list[Sample]:
    """Load image+text pairs from a Hugging Face dataset.

    `name_or_path` may be a preset key in HF_DATASETS (e.g. 'iam-lines') or any
    hub dataset id exposing an image column and a text column. `n` caps the
    number of samples (streaming, so it won't download the whole set for a
    quick check). Columns and split are auto-filled from the preset / detected.
    """
    from datasets import load_dataset

    preset = HF_DATASETS.get(name_or_path, {})
    path = preset.get("path", name_or_path)
    image_col = image_col or preset.get("image_col")
    text_col = text_col or preset.get("text_col")
    config = config or preset.get("config")
    split = split or preset.get("split", "test")

    ds = load_dataset(path, name=config, split=split, streaming=streaming)
    samples: list[Sample] = []
    for i, row in enumerate(ds):
        if n is not None and i >= n:
            break
        ic = image_col or _auto_col(row, _IMG_COL_HINTS)
        tc = text_col or _auto_col(row, _TXT_COL_HINTS)
        samples.append(Sample(name=f"{path}:{split}:{i}", gt=str(row[tc]),
                              image=row[ic]))
    return samples


def is_line_level(name_or_path: str) -> bool:
    return HF_DATASETS.get(name_or_path, {}).get("level") == "line"


def build_iam_pages(n_pages: int = 20, lines_per_page: tuple[int, int] = (4, 8),
                    max_skew_deg: float = 3.0, gap_px: int = 22, margin_px: int = 50,
                    max_indent_px: int = 60, seed: int = 0,
                    source_split: str = "test") -> list[Sample]:
    """Synthesize multi-line 'pages' by stacking real IAM line images, with
    random per-line indent and a small page skew. The skew makes naive
    top-to-bottom ordering fail, so a low CER here means `pipeline.reading_order`
    survived; WordAcc stays high since the stacking doesn't change recognition.

    Recognition CER is still IAM-optimistic (TrOCR trained on IAM); the point of
    this set is to measure detection + line clustering + ordering."""
    import cv2
    import numpy as np
    from datasets import load_dataset

    rng = np.random.default_rng(seed)
    it = iter(load_dataset("Teklia/IAM-line", split=source_split, streaming=True))
    pages: list[Sample] = []
    for p in range(n_pages):
        k = int(rng.integers(lines_per_page[0], lines_per_page[1] + 1))
        rows, texts = [], []
        for _ in range(k):
            row = next(it)
            rows.append(np.asarray(row["image"].convert("L")))
            texts.append(row["text"])
        indents = [int(rng.integers(0, max_indent_px + 1)) for _ in rows]
        width = margin_px * 2 + max(r.shape[1] + indents[i] for i, r in enumerate(rows))
        height = margin_px * 2 + sum(r.shape[0] for r in rows) + gap_px * (k - 1)
        canvas = np.full((height, width), 255, np.uint8)
        y = margin_px
        for r, ind in zip(rows, indents):
            x = margin_px + ind
            canvas[y:y + r.shape[0], x:x + r.shape[1]] = r
            y += r.shape[0] + gap_px
        if max_skew_deg:
            ang = float(rng.uniform(-max_skew_deg, max_skew_deg))
            m = cv2.getRotationMatrix2D((width / 2, height / 2), ang, 1.0)
            canvas = cv2.warpAffine(canvas, m, (width, height),
                                    borderValue=255, flags=cv2.INTER_LINEAR)
        pages.append(Sample(name=f"iam-page-{p:03d}", gt="\n".join(texts),
                            image=cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)))
    return pages


def _gnhk_words(data) -> list[dict]:
    """GNHK json is normally a list of word dicts; tolerate a dict wrapper."""
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for v in data.values():
            if isinstance(v, list):
                return v
    return []


def _gnhk_reading_order_text(words: list[dict], drop_special: bool = True,
                             keep_types: set[str] | None = None) -> str:
    """Reconstruct a page's reading-order transcript from GNHK word annotations
    (each: 'text', 'polygon' {x0,y0..x3,y3}, 'line_idx', 'type'). Groups by
    line_idx, orders lines by mean-y, words within a line by x.

    `%...%` tokens (%NA%/%math%/%SC%/...) mark non-transcribable content and are
    dropped by default — same as the GNHK recognition benchmark. `keep_types`
    optionally restricts to certain region types (e.g. {'H'} for handwritten),
    once you've seen the type codes in the real data."""
    lines: dict[int, list[tuple[float, float, str]]] = {}
    for w in words:
        t = str(w.get("text", "")).strip()
        if not t or (drop_special and t.startswith("%") and t.endswith("%")):
            continue
        if keep_types is not None and w.get("type") not in keep_types:
            continue
        poly = w.get("polygon", {}) or {}
        xs = [poly.get(f"x{i}", 0) for i in range(4)]
        ys = [poly.get(f"y{i}", 0) for i in range(4)]
        lines.setdefault(int(w.get("line_idx", -1)), []).append(
            (min(xs), min(ys), t))
    ordered = sorted(lines.values(),
                     key=lambda ws: sum(y for _, y, _ in ws) / len(ws))
    return "\n".join(" ".join(t for _, _, t in sorted(ws, key=lambda e: e[0]))
                     for ws in ordered)


def load_gnhk(folder: str | Path, drop_special: bool = True,
              keep_types: set[str] | None = None) -> list[Sample]:
    """Load GNHK (real-photo handwriting) pages from a locally downloaded folder
    (https://goodnotes.com/gnhk — agree to the CC-BY-4.0 terms, then unzip the
    train/test zips here). Expects image files with sibling .json word
    annotations; GT is reconstructed in reading order from line_idx."""
    import json
    folder = Path(folder)
    samples: list[Sample] = []
    for jf in sorted(folder.rglob("*.json")):
        img = next((c for ext in IMAGE_EXTS
                    for c in [jf.with_suffix(ext)] if c.is_file()), None)
        if img is None:
            continue
        try:
            words = _gnhk_words(json.loads(jf.read_text(encoding="utf-8")))
        except json.JSONDecodeError:
            continue
        gt = _gnhk_reading_order_text(words, drop_special, keep_types)
        if gt.strip():
            samples.append(Sample(name=jf.stem, image_path=img, gt=gt))
    return samples


# --------------------------------------------------------------------------
# scoring
# --------------------------------------------------------------------------

@dataclass
class Score:
    system: str
    n: int = 0
    cer_edits: int = 0
    cer_ref: int = 0
    wer_edits: int = 0
    wer_ref: int = 0
    wa_matched: int = 0
    wa_total: int = 0
    seconds: float = 0.0
    per_item: list[dict] = field(default_factory=list)

    @property
    def cer(self) -> float:
        return self.cer_edits / max(self.cer_ref, 1)

    @property
    def wer(self) -> float:
        return self.wer_edits / max(self.wer_ref, 1)

    @property
    def word_acc(self) -> float | None:
        # order-free fraction of reference words recovered (higher is better)
        return self.wa_matched / self.wa_total if self.wa_total else None

    @property
    def sec_per_img(self) -> float:
        return self.seconds / max(self.n, 1)


def evaluate_system(name: str, samples: list[Sample], predict_text,
                    predict_lines=None, norm: NormCfg | None = None,
                    progress: bool = False, word_metrics: bool = True) -> Score:
    """Score one system. predict_text(image_rgb)->str. WordAcc is computed
    order-free from the text, so predict_lines is not required (kept for
    backward compatibility, ignored).

    word_metrics=False skips WER and WordAcc — set it for **space-free CJK**,
    where there are no word boundaries: stripping spaces makes the whole page one
    token, so WER collapses to 100% and WordAcc to 0% (meaningless). Only CER
    (character-level) is valid there; the table then shows WER/WordAcc as '—'.

    progress=True prints a line per image (running CER + per-image seconds) so a
    slow run — e.g. heavy PaddleOCR models on a CPU runtime — shows it is working,
    not hung."""
    norm = norm or NormCfg()
    sc = Score(system=name)
    total = len(samples)
    for s in samples:
        img = s.image_rgb()
        t0 = time.perf_counter()
        hyp = predict_text(img)
        dt = time.perf_counter() - t0

        ce, cr = cer_counts(s.gt, hyp, norm)
        sc.n += 1
        sc.cer_edits += ce; sc.cer_ref += cr
        sc.seconds += dt
        item = {"system": name, "image": s.name, "cer": ce / max(cr, 1),
                "ref_chars": cr, "seconds": round(dt, 3)}
        if word_metrics:
            we, wr = wer_counts(s.gt, hyp, norm)
            wm, wt = word_acc_counts(s.gt, hyp, norm)
            sc.wer_edits += we; sc.wer_ref += wr
            sc.wa_matched += wm; sc.wa_total += wt
            item.update(wer=we / max(wr, 1), word_acc=wm / max(wt, 1))
        sc.per_item.append(item)
        if progress:
            print(f"  [{name}] {sc.n}/{total} {s.name}: cer={sc.cer:.3f} "
                  f"({dt:.1f}s)", flush=True)
    return sc


# --------------------------------------------------------------------------
# systems under test (lazy imports — none required unless used)
# --------------------------------------------------------------------------

def recognizer_predictor(recognizer=None, **rec_kwargs):
    """predict_text using ONLY the TrOCR recognizer (no detection/ordering).
    Use on single-line images (e.g. IAM lines) for a recognizer-quality number
    directly comparable to published TrOCR CER."""
    if recognizer is None:
        from ocr_engine import Recognizer
        recognizer = Recognizer(**rec_kwargs)
    return lambda img: recognizer([img])[0]


def evaluate_recognizer(name: str, samples: list[Sample], recognizer=None,
                        norm: NormCfg | None = None, batch_size: int = 16) -> Score:
    """Fast batched recognizer-only scoring for line datasets — runs the whole
    set through TrOCR in batches instead of one image at a time."""
    norm = norm or NormCfg()
    if recognizer is None:
        from ocr_engine import Recognizer
        recognizer = Recognizer()
    imgs = [s.image_rgb() for s in samples]
    t0 = time.perf_counter()
    hyps: list[str] = []
    for i in range(0, len(imgs), batch_size):
        hyps.extend(recognizer(imgs[i:i + batch_size], batch_size=batch_size))
    dt = time.perf_counter() - t0

    sc = Score(system=name, seconds=dt)
    for s, hyp in zip(samples, hyps):
        ce, cr = cer_counts(s.gt, hyp, norm)
        we, wr = wer_counts(s.gt, hyp, norm)
        wm, wt = word_acc_counts(s.gt, hyp, norm)
        sc.n += 1
        sc.cer_edits += ce; sc.cer_ref += cr
        sc.wer_edits += we; sc.wer_ref += wr
        sc.wa_matched += wm; sc.wa_total += wt
        sc.per_item.append({"system": name, "image": s.name, "cer": ce / max(cr, 1),
                            "wer": we / max(wr, 1), "word_acc": wm / max(wt, 1),
                            "ref_chars": cr})
    return sc


def pipeline_predictors(engine=None, **engine_kwargs):
    """Return (predict_text, predict_lines) for THIS project's pipeline."""
    if engine is None:
        from ocr_engine import OcrEngine
        engine = OcrEngine(**engine_kwargs)
    cache: dict[int, dict] = {}

    def _run(img):
        key = id(img)
        if key not in cache:
            cache.clear()
            cache[key] = engine.run(img)
        return cache[key]

    return (lambda img: _run(img)["text"],
            lambda img: [l["text"] for l in _run(img)["lines"]])


def baseline_predict(name: str):
    """Return predict_text for a baseline OCR system (lazy, raises if missing)."""
    name = name.lower()
    if name == "tesseract":
        import pytesseract
        from PIL import Image
        return lambda img: pytesseract.image_to_string(Image.fromarray(img))
    if name == "easyocr":
        import easyocr
        reader = easyocr.Reader(["en"], gpu=True)
        return lambda img: "\n".join(reader.readtext(img, detail=0, paragraph=True))
    if name == "paddleocr":
        import cv2
        from paddleocr import PaddleOCR
        # enable_mkldnn=False avoids paddle's PIR+oneDNN crash
        # (ConvertPirAttribute2RuntimeAttribute), same fix Detector uses.
        ocr = None
        for kw in (dict(lang="en", enable_mkldnn=False, use_doc_orientation_classify=False,
                        use_doc_unwarping=False, use_textline_orientation=False),
                   dict(lang="en", enable_mkldnn=False),
                   dict(use_angle_cls=False, lang="en", enable_mkldnn=False),  # 2.x
                   dict(lang="en")):
            try:
                ocr = PaddleOCR(**kw)
                break
            except TypeError:
                continue
        if ocr is None:
            raise RuntimeError("could not initialize PaddleOCR")
        def _run(img):
            bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            if hasattr(ocr, "predict"):                      # 3.x: list of dict-like
                try:
                    out = []
                    for r in ocr.predict(bgr):
                        texts = r.get("rec_texts") if hasattr(r, "get") else None
                        if texts:
                            out.extend(texts)
                    return "\n".join(out)
                except Exception:
                    pass
            res = ocr.ocr(bgr)                               # 2.x: [[[box,(text,score)],...]]
            out = []
            for page in res or []:
                for line in page or []:
                    try:
                        out.append(line[1][0])
                    except (IndexError, TypeError):
                        pass
            return "\n".join(out)
        return _run
    raise ValueError(f"unknown baseline: {name}")


# --------------------------------------------------------------------------
# reporting
# --------------------------------------------------------------------------

def format_table(scores: list[Score]) -> str:
    # CER/WER are the headline (lower better); WordAcc is an order-free
    # recognition diagnostic (higher better) — NOT subtracted from CER.
    head = f"{'system':<16}{'CER':>8}{'WER':>8}{'WordAcc':>9}{'sec/img':>9}{'n':>5}"
    rows = [head, "-" * len(head)]
    for s in sorted(scores, key=lambda x: x.cer):
        # WER/WordAcc are '—' when not computed (e.g. space-free CJK: no words)
        wer = f"{s.wer:>8.1%}" if s.wer_ref else f"{'—':>8}"
        wa = f"{s.word_acc:>9.1%}" if s.word_acc is not None else f"{'—':>9}"
        rows.append(f"{s.system:<16}{s.cer:>8.1%}{wer}{wa}"
                    f"{s.sec_per_img:>9.2f}{s.n:>5}")
    return "\n".join(rows)


def save_csv(scores: list[Score], path: str | Path) -> None:
    import csv
    rows = [it for s in scores for it in s.per_item]
    if not rows:
        return
    keys = ["system", "image", "cer", "wer", "word_acc", "ref_chars", "seconds"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    src = ap.add_mutually_exclusive_group(required=True)
    src.add_argument("--data", help="folder of images + sibling .txt ground truth")
    src.add_argument("--hf", help=f"HF dataset: preset ({', '.join(HF_DATASETS)}) or any hub id")
    ap.add_argument("--split", default="test", help="HF split (default: test)")
    ap.add_argument("--n", type=int, default=None, help="cap number of HF samples (quick check)")
    ap.add_argument("--config", default=None, help="HF dataset config/subset name")
    ap.add_argument("--mode", choices=["auto", "recognizer", "pipeline"], default="auto",
                    help="auto: recognizer for line datasets, full pipeline otherwise")
    ap.add_argument("--baselines", default="", help="comma list: tesseract,easyocr,paddleocr")
    ap.add_argument("--csv", default="eval_results.csv")
    ap.add_argument("--lowercase", action="store_true")
    ap.add_argument("--strip-punct", action="store_true")
    args = ap.parse_args()

    if args.hf:
        samples = load_hf(args.hf, split=args.split, n=args.n, config=args.config)
        line_level = is_line_level(args.hf)
        print(f"loaded {len(samples)} samples from HF {args.hf} [{args.split}]")
    else:
        samples = load_pairs(args.data)
        line_level = False
        print(f"loaded {len(samples)} samples from {args.data}")
    if not samples:
        print("No (image, ground-truth) pairs found. See module docstring for layout.")
        return 1

    norm = NormCfg(lowercase=args.lowercase, strip_punct=args.strip_punct)
    use_recognizer = args.mode == "recognizer" or (args.mode == "auto" and line_level)

    scores: list[Score] = []
    if use_recognizer:
        print("scoring recognizer only (no detection/ordering)")
        scores.append(evaluate_recognizer("trocr-recognizer", samples, norm=norm))
    else:
        pt, pl = pipeline_predictors()
        scores.append(evaluate_system("this-pipeline", samples, pt, pl, norm))

    for b in [x.strip() for x in args.baselines.split(",") if x.strip()]:
        try:
            scores.append(evaluate_system(b, samples, baseline_predict(b), norm=norm))
        except Exception as e:  # missing dep / runtime error in a baseline
            print(f"  (skip baseline {b}: {type(e).__name__}: {e})")

    print("\n" + format_table(scores))
    save_csv(scores, args.csv)
    print(f"\nper-image breakdown -> {args.csv}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### 9a. Metric self-tests (proves the math; no data needed)

In [ ]:
!python -m unittest discover -s . -p 'test_metrics.py' -v

### 9b. Score this pipeline on your labeled data
Set `EVAL_DIR`, drop image+`.txt` pairs in it, run.

In [ ]:
import evaluate as E
from pathlib import Path

EVAL_DIR = '/content/eval_data'
Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)

# normalization applied to BOTH prediction and ground truth before scoring:
NORM = E.NormCfg(lowercase=False, strip_punct=False)  # standard CER

samples = E.load_pairs(EVAL_DIR)
print(f'{len(samples)} labeled sample(s) in {EVAL_DIR}')
assert samples, ('No image+.txt pairs found. Drag e.g. page1.jpg AND page1.txt '
                 'into the eval_data folder, then re-run.')

pt, _ = E.pipeline_predictors(engine)          # reuse the engine from section 5
scores = [E.evaluate_system('this-pipeline', samples, pt, norm=NORM)]
print(); print(E.format_table(scores))

### 9c. Add baselines (Tesseract / EasyOCR / PaddleOCR) on the same data
Installs are optional; skip any you don't want. Then they appear in the same table.

In [ ]:
# Tesseract + EasyOCR (PaddleOCR is already installed from section 1)
!apt-get -qq install -y tesseract-ocr >/dev/null && pip -q install pytesseract easyocr

BASELINES = ['tesseract', 'easyocr', 'paddleocr']   # trim as you like
for b in BASELINES:
    try:
        scores.append(E.evaluate_system(b, samples, E.baseline_predict(b), norm=NORM))
        print(f'scored {b}')
    except Exception as ex:
        print(f'skip {b}: {type(ex).__name__}: {ex}')

print(); print(E.format_table(scores))
E.save_csv(scores, 'eval_results.csv')
print('\nper-image breakdown saved to eval_results.csv')

In [ ]:
# bar chart of CER/WER per system
import matplotlib.pyplot as plt
import numpy as np
names = [s.system for s in scores]
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(1.6 * len(names) + 3, 4))
ax.bar(x - 0.2, [s.cer for s in scores], 0.4, label='CER')
ax.bar(x + 0.2, [s.wer for s in scores], 0.4, label='WER')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('error rate'); ax.legend(); ax.set_title('Lower is better')
ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

## 10. Score on a public dataset (IAM) — a citable number
No registration or manual labeling: `evaluate.load_hf` streams **`Teklia/IAM-line`** (IAM
handwriting test lines, image+text, not gated) straight from Hugging Face. IAM lines are
single lines, so this scores the **recognizer alone** (no detection/ordering) — directly
comparable to the published **TrOCR-large CER ≈ 2.9%**.

Start with `N = 200` for a ~1-minute check; set `N = None` for the full 2,920-line test set
(a few minutes on a T4).

In [ ]:
%pip install -q datasets
import evaluate as E

N = 200          # None = full IAM test (2,920 lines)
NORM = E.NormCfg(lowercase=False, strip_punct=False)   # standard CER

iam = E.load_hf('iam-lines', split='test', n=N)
print(f'loaded {len(iam)} IAM test lines')
print('example GT:', repr(iam[0].gt))

# reuse the recognizer already loaded in section 5 (engine.recognizer)
iam_scores = [E.evaluate_recognizer('trocr-recognizer', iam,
                                    recognizer=engine.recognizer, norm=NORM)]
print(); print(E.format_table(iam_scores))
print('\n(compare CER to the published TrOCR-large IAM number, ~2.9%)')

### 10a. (optional) Baselines on the same IAM lines
Same images, other engines — the only fair comparison. Skip if you only wanted your number.

In [ ]:
# needs the installs from section 9c (tesseract/easyocr); paddleocr already present
for b in ['tesseract', 'easyocr', 'paddleocr']:
    try:
        iam_scores.append(E.evaluate_system(b, iam, E.baseline_predict(b), norm=NORM))
        print(f'scored {b}')
    except Exception as ex:
        print(f'skip {b}: {type(ex).__name__}: {ex}')
print(); print(E.format_table(iam_scores))
E.save_csv(iam_scores, 'iam_results.csv')

## 11. Full-page evaluation (your pipeline's actual job)
IAM lines (section 10) only score the recognizer. These score the **whole pipeline** —
detection → reading order → recognition — on multi-line images, using
`E.pipeline_predictors(engine)`. Compare **CER** (penalizes wrong reading order) against
**WordAcc** (order-free): high WordAcc with high CER means recognition is fine but lines are
out of order.

These IAM-based pages (11a, 11b) are quick **detection + reading-order** checks. Your headline
resume comparison vs other OCRs is **Section 12 (GNHK)** — real photos, unbiased.

- **11a IAM_Sentences** — real multi-line IAM images, streamed from HF. One line of setup.
- **11b Synthetic IAM pages** — stacks real IAM lines into skewed, indented pages; the skew
  makes naive top-to-bottom ordering fail, so it genuinely tests reading order.

Caveat baked in: TrOCR was trained on IAM, so IAM recognition CER is optimistic. IAM still
validly tests detection+ordering (layout was never trained); GNHK (§12) gives the unbiased number.

### 11a. IAM_Sentences — real multi-line images (HF)

In [ ]:
import evaluate as E
NORM = E.NormCfg(lowercase=False, strip_punct=False)

PAGES_N = 30                       # cap for a quick run; raise for a fuller number
sent = E.load_hf('iam-sentences', n=PAGES_N)
print(f'loaded {len(sent)} multi-line IAM images')

pt, _ = E.pipeline_predictors(engine)      # full detect -> order -> recognize
page_scores = [E.evaluate_system('this-pipeline', sent, pt, norm=NORM)]
print(); print(E.format_table(page_scores))
print('\nCER = with reading order; WordAcc = order-free recognition.')

### 11b. Synthetic IAM pages — the reading-order stress test
Real IAM lines stacked into skewed, indented pages. The skew makes naive top-to-bottom
ordering fail, so a low CER here means reading order survived; WordAcc should stay high
throughout (recognition is unaffected by the stacking).

In [ ]:
import matplotlib.pyplot as plt
pages = E.build_iam_pages(n_pages=20, lines_per_page=(4, 8), max_skew_deg=3.0, seed=0)
print(f'built {len(pages)} synthetic pages')
plt.figure(figsize=(10, 6)); plt.imshow(pages[0].image_rgb())
plt.axis('off'); plt.title(pages[0].name + '  (GT lines: ' + str(pages[0].gt.count(chr(10)) + 1) + ')')
plt.show()

pt, _ = E.pipeline_predictors(engine)
syn_scores = [E.evaluate_system('this-pipeline', pages, pt, norm=NORM)]
print(); print(E.format_table(syn_scores))
print('\nlow CER = reading order survives the skew | high WordAcc = recognition fine')

## 12. GNHK head-to-head — the resume benchmark
Real phone-photo English handwriting, **out-of-domain** for TrOCR, so this CER is unbiased —
the honest number to put on a resume. Your pipeline runs against **Tesseract** and the full
**PaddleOCR** on the *same* pages under one metric.

**One-time download (manual — GNHK is behind a CC-BY-4.0 click-through, no direct link):**
1. Open <https://goodnotes.com/gnhk>, scroll down, accept the terms, download `train_data.zip`
   and/or `test_data.zip`.
2. Upload the zip(s) to Colab (📁 sidebar → into `/content`), **or** drop them in Google Drive
   and `from google.colab import drive; drive.mount('/content/drive')`.
3. Run the cells below — they unzip, rebuild reading-order GT, sanity-check it, then score.

In [ ]:
# install Tesseract (PaddleOCR already present from section 1)
!apt-get -qq install -y tesseract-ocr >/dev/null && pip -q install pytesseract

import glob, zipfile, pathlib, evaluate as E
NORM = E.NormCfg(lowercase=False, strip_punct=False)
GNHK_DIR = '/content/gnhk'

# unzip whatever GNHK zip you uploaded (archive.zip, train_data.zip, ...)
for z in glob.glob('/content/*.zip'):
    with zipfile.ZipFile(z) as zf: zf.extractall(GNHK_DIR)
    print('unzipped', z)

# show what we actually got, so any layout problem is obvious
root = pathlib.Path(GNHK_DIR)
jsons = list(root.rglob('*.json'))
imgs = [p for p in root.rglob('*') if p.suffix.lower() in E.IMAGE_EXTS]
print(f'found {len(jsons)} json + {len(imgs)} image files under {GNHK_DIR}')
if jsons: print('  e.g. json :', jsons[0].relative_to(root))
if imgs:  print('  e.g. image:', imgs[0].relative_to(root))

gnhk = E.load_gnhk(GNHK_DIR)
print(f'{len(gnhk)} GNHK pages loaded')
assert gnhk, ('0 pages. If counts above are >0, image+json do not share a folder/stem; else the zip had no GNHK .json files.')

### 12a. Sanity-check the reconstructed ground truth
GNHK GT is rebuilt from per-word `line_idx`; eyeball one page so you trust the reference text
before scoring against it.

In [ ]:
import matplotlib.pyplot as plt
s = gnhk[0]
plt.figure(figsize=(11, 8)); plt.imshow(s.image_rgb()); plt.axis('off'); plt.title(s.name); plt.show()
print('--- reconstructed reading-order ground truth ---')
print(s.gt)

### 12b. Score: your pipeline vs Tesseract vs PaddleOCR
Evaluates on the official **test** split (the standard set; scoring train+test would be hours).
**CER/WER** are the headline comparison; **WordAcc** (order-free) tells you whether errors are
recognition or ordering. Set `GNHK_N` to an int to cap for a quick pass; `None` runs the full split.

In [ ]:
import pathlib, evaluate as E
NORM = E.NormCfg(lowercase=False, strip_punct=False)

# use the official TEST split if present under GNHK_DIR
test_dirs = [p for p in pathlib.Path(GNHK_DIR).rglob('*')
             if p.is_dir() and 'test' in p.name.lower()]
EVAL_ROOT = str(test_dirs[0]) if test_dirs else GNHK_DIR
eval_pages = E.load_gnhk(EVAL_ROOT)

GNHK_N = None                       # None = all pages in EVAL_ROOT; set an int to cap
subset = eval_pages[:GNHK_N] if GNHK_N else eval_pages
print(f'evaluating on {EVAL_ROOT}: {len(subset)} pages '
      f'(~{len(subset) * 14 / 60:.0f} min for the pipeline alone on a T4)')

pt, _ = E.pipeline_predictors(engine)               # full detect -> order -> recognize
gnhk_scores = [E.evaluate_system('this-pipeline', subset, pt, norm=NORM)]
print('scored this-pipeline')
for b in ['tesseract', 'paddleocr']:                # your chosen competitors
    try:
        gnhk_scores.append(E.evaluate_system(b, subset, E.baseline_predict(b), norm=NORM))
        print('scored', b)
    except Exception as ex:
        import traceback
        print('skip', b, ':', type(ex).__name__, ex); traceback.print_exc()

print(); print(E.format_table(gnhk_scores))
E.save_csv(gnhk_scores, 'gnhk_results.csv')
print('\nper-image breakdown -> gnhk_results.csv')

In [ ]:
# resume chart: CER/WER per system on GNHK
import numpy as np, matplotlib.pyplot as plt
names = [s.system for s in gnhk_scores]
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(1.7 * len(names) + 3, 4))
ax.bar(x - 0.2, [s.cer for s in gnhk_scores], 0.4, label='CER')
ax.bar(x + 0.2, [s.wer for s in gnhk_scores], 0.4, label='WER')
for i, s in enumerate(gnhk_scores):
    ax.text(i - 0.2, s.cer, f'{s.cer:.0%}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('error rate (lower is better)'); ax.legend()
ax.set_title('GNHK: full-page handwriting OCR comparison'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('gnhk_comparison.png', dpi=130); plt.show()
print('saved gnhk_comparison.png — drop this straight into your resume/README')

### Any other dataset
`E.load_hf('org/dataset', n=100, image_col=..., text_col=...)` works for any hub dataset with
an image + text column. Score lines with `E.evaluate_recognizer`, pages with the
`pipeline_predictors` + `evaluate_system` pattern above. Only compare systems on the **same** set.

## Debugging guide
If something looks wrong, report back with the panel image + recognized text and note which case it is:

1. **Boxes miss/split handwriting** → detection problem (we tune `unclip_ratio`/`box_thresh` or det model).
2. **Boxes fine, numbers in wrong sequence** → ordering problem (send the panel; the geometry is testable, we add your case as a unit test).
3. **Boxes and order fine, words wrong** → recognition problem (TrOCR domain gap → fine-tuning / beams). **Low WordAcc** confirms this; **high WordAcc with high CER** means ordering instead.
4. **Page rotated 90°/180°** → known limitation, orientation classifier is on the roadmap.
